# STEP 4 — Clustering from corrected directed DGI embeddings


In [ ]:
# ------------------------------------------------------------
# 0. Imports
# ------------------------------------------------------------

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display
from scipy.optimize import linear_sum_assignment

from sklearn.cluster import KMeans, SpectralClustering
from sklearn.feature_selection import mutual_info_classif
from sklearn.manifold import TSNE
from sklearn.metrics import (
    adjusted_mutual_info_score,
    adjusted_rand_score,
    balanced_accuracy_score,
    calinski_harabasz_score,
    classification_report,
    completeness_score,
    davies_bouldin_score,
    homogeneity_score,
    normalized_mutual_info_score,
    silhouette_score,
    v_measure_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.tree import (
    DecisionTreeClassifier,
    export_text,
    plot_tree,
)

In [ ]:
# ------------------------------------------------------------
# 1. Paths and MANUAL clustering configuration
# ------------------------------------------------------------

EXPECTED_DGI_FOLDER = Path("cora_directed_dgi_ard")

ARD_PATH = Path(
    "ard_matrix_cora_directed_out.csv"
)

MEMBERSHIP_PATH = Path(
    "membership_matrix_cora_directed.csv"
)

# Standardized directed edge list generated by the corrected Step 0:
# source = citing paper, target = cited paper.
DIRECTED_EDGES_PATH = Path(
    "cora_directed_edges.csv"
)

OUTPUT_FOLDER = Path(
    "cora_directed_clustering_training_embeddings_comparison"
)

SAVE_OUTPUTS = True
STANDARDIZE_EMBEDDINGS = False
RANDOM_STATE = 123


# ============================================================
# MANUAL CONFIGURATION
# ============================================================
#
# For EACH approach, choose independently:
#
#   method: "kmeans", "spectral", or "gmm"
#   k:      any integer >= 2
#
# You only need to edit this dictionary when you want to test
# another clustering method or another number of clusters.
# ============================================================

FINAL_CLUSTERING_CONFIG = {
    "original_directed_graph": {
        "method": "kmeans",
        "k": 7,
    },

    "multiple_encoders_directed": {
        "method": "kmeans",
        "k": 7,
    },

    "one_encoder_directed": {
        "method": "kmeans",
        "k": 7,
    },

    "one_encoder_training_directed": {
        "method": "kmeans",
        "k": 7,
    },

    "batch_directed": {
        "method": "kmeans",
        "k": 7,
    },

    "batch_training_directed": {
        "method": "kmeans",
        "k": 7,
    },
}


# ============================================================
# OPTIONAL DIAGNOSTIC SWEEP
# ============================================================
#
# This is independent from FINAL_CLUSTERING_CONFIG.
#
# Example:
#   DIAGNOSTIC_K_VALUES = [2, 3, 4, 5, 6, 7, 8]
#
# If you only want the manually selected final analyses,
# set RUN_DIAGNOSTICS = False.
# ============================================================

RUN_DIAGNOSTICS = True

DIAGNOSTIC_METHODS = [
    "kmeans",
    "gmm",
    "spectral",
]

DIAGNOSTIC_K_VALUES = list(
    range(2, 9)
)


# ------------------------------------------------------------
# Validate manual configuration
# ------------------------------------------------------------

VALID_CLUSTERING_METHODS = {
    "kmeans",
    "gmm",
    "spectral",
}

EXPECTED_APPROACHES = {
    "original_directed_graph",
    "multiple_encoders_directed",
    "one_encoder_directed",
    "one_encoder_training_directed",
    "batch_directed",
    "batch_training_directed",
}

if set(FINAL_CLUSTERING_CONFIG) != EXPECTED_APPROACHES:
    raise ValueError(
        "FINAL_CLUSTERING_CONFIG must contain exactly these approaches: "
        f"{sorted(EXPECTED_APPROACHES)}"
    )

for approach_name, config in FINAL_CLUSTERING_CONFIG.items():
    method_name = str(
        config["method"]
    ).lower()

    n_clusters = int(
        config["k"]
    )

    if method_name not in VALID_CLUSTERING_METHODS:
        raise ValueError(
            f"{approach_name}: unknown method '{method_name}'. "
            "Use 'kmeans', 'gmm', or 'spectral'."
        )

    if n_clusters < 2:
        raise ValueError(
            f"{approach_name}: k must be at least 2."
        )

    FINAL_CLUSTERING_CONFIG[
        approach_name
    ]["method"] = method_name

    FINAL_CLUSTERING_CONFIG[
        approach_name
    ]["k"] = n_clusters


if SAVE_OUTPUTS:
    OUTPUT_FOLDER.mkdir(
        parents=True,
        exist_ok=True
    )


print("ARD path:", ARD_PATH)
print("Membership path:", MEMBERSHIP_PATH)
print("Directed edge path:", DIRECTED_EDGES_PATH)
print("Output folder:", OUTPUT_FOLDER)

print("\n=== MANUAL FINAL CLUSTERING CONFIGURATION ===")

for approach_name, config in FINAL_CLUSTERING_CONFIG.items():
    print(
        f"{approach_name:30s} "
        f"method={config['method']:8s} "
        f"k={config['k']}"
    )

print("\nRun diagnostics:", RUN_DIAGNOSTICS)

if RUN_DIAGNOSTICS:
    print(
        "Diagnostic methods:",
        DIAGNOSTIC_METHODS
    )
    print(
        "Diagnostic k values:",
        DIAGNOSTIC_K_VALUES
    )


In [ ]:
# ------------------------------------------------------------
# 2. Locate and validate the corrected directed Step 3 output
# ------------------------------------------------------------

def locate_directed_dgi_folder(
    expected_folder,
    expected_embedding_name
):
    expected_embedding_path = (
        expected_folder
        / expected_embedding_name
    )

    if expected_embedding_path.exists():
        return expected_folder

    matching_files = list(
        Path(".").rglob(
            expected_embedding_name
        )
    )

    if len(matching_files) == 1:
        return matching_files[0].parent

    if len(matching_files) == 0:
        print("Current working directory:")
        print(Path.cwd())

        print("\nAvailable .npy files:")
        for path in Path(".").rglob("*.npy"):
            print(path)

        raise FileNotFoundError(
            "The corrected directed Step 3 embeddings were not found. "
            "Run the corrected Step 3 notebook first or "
            "set DGI_FOLDER manually."
        )

    print("Multiple possible directed DGI folders were found:")

    for path in matching_files:
        print(path.parent)

    raise ValueError(
        "Set DGI_FOLDER manually to the correct Step 3 folder."
    )


DGI_FOLDER = locate_directed_dgi_folder(
    expected_folder=EXPECTED_DGI_FOLDER,
    expected_embedding_name=(
        "embeddings_original_directed_ard_out.npy"
    )
)

NODE_ORDER_PATH = (
    DGI_FOLDER
    / "node_order.csv"
)

CONFIGURATION_PATH = (
    DGI_FOLDER
    / "directed_dgi_configuration.json"
)


def validate_step3_configuration(
    configuration_path
):
    if not configuration_path.exists():
        raise FileNotFoundError(
            "Missing directed_dgi_configuration.json. "
            "Run the corrected Step 3 notebook before Step 4."
        )

    with open(
        configuration_path,
        "r",
        encoding="utf-8"
    ) as file:
        configuration = json.load(file)

    encoder_text = str(
        configuration.get("encoder", "")
    ).lower()

    feature_file = str(
        configuration.get("feature_file", "")
    )

    if "incoming" not in encoder_text or "outgoing" not in encoder_text:
        raise ValueError(
            "The Step 3 configuration does not describe the "
            "directed incoming/outgoing encoder."
        )

    if "ard_matrix_cora_directed_out.csv" not in feature_file:
        raise ValueError(
            "The Step 3 configuration does not use the expected "
            "outgoing ARD file."
        )

    return configuration


step3_configuration = validate_step3_configuration(
    CONFIGURATION_PATH
)

print("Directed Step 3 folder:", DGI_FOLDER)
print("Node order path:", NODE_ORDER_PATH)
print("Step 3 configuration:", CONFIGURATION_PATH)
print(
    "Step 3 encoder:",
    step3_configuration.get("encoder")
)
print(
    "Step 3 feature file:",
    step3_configuration.get("feature_file")
)


In [ ]:
# ------------------------------------------------------------
# 3. General save helpers
# ------------------------------------------------------------

def maybe_save_csv(
    obj,
    output_path,
    **kwargs
):
    if not SAVE_OUTPUTS:
        return

    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    obj.to_csv(
        output_path,
        **kwargs
    )


def maybe_save_figure(
    output_path,
    dpi=300
):
    if not SAVE_OUTPUTS:
        return

    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    plt.savefig(
        output_path,
        dpi=dpi,
        bbox_inches="tight"
    )

In [ ]:
# ------------------------------------------------------------
# 4. Load embeddings and node matrices
# ------------------------------------------------------------

def load_embeddings(
    embeddings_path,
    standardize=False
):
    embeddings_path = Path(
        embeddings_path
    )

    if not embeddings_path.exists():
        raise FileNotFoundError(
            f"Missing embedding file: {embeddings_path}"
        )

    embeddings = np.load(
        embeddings_path
    )

    if embeddings.ndim != 2:
        raise ValueError(
            f"{embeddings_path.name} must contain a 2D matrix."
        )

    if not np.isfinite(embeddings).all():
        raise ValueError(
            f"{embeddings_path.name} contains NaN or infinity."
        )

    embeddings = embeddings.astype(
        np.float64,
        copy=False
    )

    if standardize:
        embeddings = StandardScaler().fit_transform(
            embeddings
        )

    return embeddings


def load_node_order(
    node_order_path
):
    node_order_path = Path(
        node_order_path
    )

    if not node_order_path.exists():
        return None

    node_order_df = pd.read_csv(
        node_order_path
    )

    if "node_id" not in node_order_df.columns:
        raise ValueError(
            "node_order.csv must contain a node_id column."
        )

    node_order = (
        node_order_df["node_id"]
        .astype(int)
        .tolist()
    )

    if len(node_order) != len(set(node_order)):
        raise ValueError(
            "node_order.csv contains duplicated node IDs."
        )

    return node_order


def load_node_matrix(
    path,
    reference_node_ids=None,
    matrix_name="matrix"
):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Missing {matrix_name} file: {path}"
        )

    df = pd.read_csv(
        path,
        index_col=0
    )

    df.index = df.index.astype(int)
    df.index.name = "node_id"

    if df.index.duplicated().any():
        raise ValueError(
            f"{matrix_name} contains duplicated node IDs."
        )

    if df.isna().any().any():
        raise ValueError(
            f"{matrix_name} contains missing values."
        )

    if reference_node_ids is not None:
        reference_node_ids = [
            int(node_id)
            for node_id in reference_node_ids
        ]

        missing_nodes = (
            set(reference_node_ids)
            - set(df.index)
        )

        extra_nodes = (
            set(df.index)
            - set(reference_node_ids)
        )

        if missing_nodes:
            raise ValueError(
                f"{matrix_name} is missing nodes: "
                f"{sorted(missing_nodes)[:10]}"
            )

        if extra_nodes:
            raise ValueError(
                f"{matrix_name} contains extra nodes: "
                f"{sorted(extra_nodes)[:10]}"
            )

        df = df.loc[
            reference_node_ids
        ]

    return df

In [ ]:
# ------------------------------------------------------------
# 5. Validate Cora class membership
# ------------------------------------------------------------

def validate_exclusive_class_membership(
    class_membership_df
):
    if class_membership_df.shape[1] == 0:
        raise ValueError(
            "Class membership matrix has no columns."
        )

    values = class_membership_df.to_numpy()

    if not np.isin(values, [0, 1]).all():
        raise ValueError(
            "Cora class membership must contain only 0 and 1."
        )

    row_sums = class_membership_df.sum(
        axis=1
    )

    if not (row_sums == 1).all():
        invalid_nodes = (
            row_sums[
                row_sums != 1
            ]
            .index
            .tolist()[:10]
        )

        raise ValueError(
            "Each Cora paper must belong to exactly one class. "
            f"Invalid nodes: {invalid_nodes}"
        )

    print(
        "Every paper belongs to exactly one class:",
        True
    )

    return class_membership_df


def membership_to_class_labels(
    class_membership_df
):
    class_names = (
        class_membership_df.columns
        .astype(str)
        .tolist()
    )

    class_positions = np.argmax(
        class_membership_df.to_numpy(),
        axis=1
    )

    class_labels = np.array(
        [
            class_names[position]
            for position in class_positions
        ]
    )

    return class_labels

In [ ]:
# ------------------------------------------------------------
# 6. Embedding registry
# ------------------------------------------------------------

# IMPORTANT:
# Only explicit directed/outgoing-ARD embedding files are accepted.

dataset_registry = {
    "original_directed_graph": {
        "label": "Original directed graph",
        "embedding_path": (
            DGI_FOLDER
            / "embeddings_original_directed_ard_out.npy"
        ),
    },

    "multiple_encoders_directed": {
        "label": "Multiple Encoders",
        "embedding_path": (
            DGI_FOLDER
            / "embeddings_method1_multiple_encoders_directed_ard_out.npy"
        ),
    },

    "one_encoder_directed": {
        "label": "One Encoder — Post-training",
        "embedding_path": (
            DGI_FOLDER
            / "embeddings_method2_one_encoder_directed_ard_out.npy"
        ),
    },

    "one_encoder_training_directed": {
        "label": "One Encoder — Training-time",
        "embedding_path": (
            DGI_FOLDER
            / "embeddings_method2_one_encoder_TRAINING_MEAN_directed_ard_out.npy"
        ),
    },

    "batch_directed": {
        "label": "Full Batch — Post-training",
        "embedding_path": (
            DGI_FOLDER
            / "embeddings_method3_batch_directed_ard_out.npy"
        ),
    },

    "batch_training_directed": {
        "label": "Full Batch — Training-time",
        "embedding_path": (
            DGI_FOLDER
            / "embeddings_method3_batch_TRAINING_MEAN_directed_ard_out.npy"
        ),
    },
}

print("=== Directed embedding registry ===")

for dataset_name, dataset_info in dataset_registry.items():
    embedding_path = dataset_info["embedding_path"]

    if not embedding_path.exists():
        raise FileNotFoundError(
            f"Missing Step 3 embedding: {embedding_path}"
        )

    print(
        dataset_name,
        "->",
        embedding_path
    )


In [ ]:
# ------------------------------------------------------------
# 7. Load, align, and validate interpretation data
# ------------------------------------------------------------

step3_node_order = load_node_order(
    NODE_ORDER_PATH
)

if step3_node_order is None:
    raise FileNotFoundError(
        "node_order.csv is required to guarantee alignment "
        "with the corrected Step 3 embeddings."
    )

ard_input_df = load_node_matrix(
    path=ARD_PATH,
    reference_node_ids=step3_node_order,
    matrix_name="outgoing ARD"
)

node_order = step3_node_order

class_membership_df = load_node_matrix(
    path=MEMBERSHIP_PATH,
    reference_node_ids=node_order,
    matrix_name="Cora class membership"
)

class_membership_df = (
    validate_exclusive_class_membership(
        class_membership_df
    )
)

ground_truth_labels = (
    membership_to_class_labels(
        class_membership_df
    )
)

# ------------------------------------------------------------
# Critical Step 0 / Step 4 consistency check
# ------------------------------------------------------------
# The corrected Step 0 edge list is:
# source = citing paper, target = cited paper.
#
# Since the Cora classes are exhaustive and mutually exclusive:
#
#     sum_k ARD_out[i,k] = out_degree(i)
#
# must hold for every paper i.
if not DIRECTED_EDGES_PATH.exists():
    raise FileNotFoundError(
        "Missing cora_directed_edges.csv. "
        "Run the corrected Step 0 notebook before Step 4."
    )

directed_edges_df = pd.read_csv(
    DIRECTED_EDGES_PATH,
    sep=",",
    header=None,
    names=["source", "target", "weight"]
)

directed_edges_df["source"] = (
    directed_edges_df["source"].astype(int)
)

directed_edges_df["target"] = (
    directed_edges_df["target"].astype(int)
)

G_orientation_check = nx.DiGraph()
G_orientation_check.add_nodes_from(
    node_order
)
G_orientation_check.add_edges_from(
    directed_edges_df[
        ["source", "target"]
    ].itertuples(
        index=False,
        name=None
    )
)

ard_row_sums = (
    ard_input_df.sum(axis=1).astype(int)
)

true_out_degrees = pd.Series(
    dict(G_orientation_check.out_degree()),
    dtype=int
).reindex(node_order)

orientation_mismatch = (
    ard_row_sums != true_out_degrees
)

if orientation_mismatch.any():
    mismatch_nodes = orientation_mismatch[
        orientation_mismatch
    ].index.tolist()

    example_nodes = mismatch_nodes[:10]

    comparison = pd.DataFrame({
        "ard_row_sum": ard_row_sums.loc[example_nodes],
        "directed_out_degree": true_out_degrees.loc[example_nodes],
    })

    display(comparison)

    raise ValueError(
        "Outgoing ARD does not match the out-degree of the corrected "
        "directed Cora graph. Check that Steps 0 and 3 were rerun."
    )

if int(ard_row_sums.sum()) != G_orientation_check.number_of_edges():
    raise ValueError(
        "Total outgoing ARD counts do not equal the number "
        "of corrected directed edges."
    )

print("Outgoing ARD shape:", ard_input_df.shape)
print(
    "Class-membership shape:",
    class_membership_df.shape
)
print(
    "Number of ground-truth classes:",
    len(np.unique(ground_truth_labels))
)
print(
    "Orientation check passed: "
    "ARD row sums = corrected graph out-degrees."
)
print(
    "Citation convention confirmed: citing -> cited."
)

display(ard_input_df.head())
display(class_membership_df.head())


In [ ]:
# ------------------------------------------------------------
# 8. Load and validate all directed embeddings
# ------------------------------------------------------------

def load_dataset_from_registry(
    dataset_name
):
    dataset_info = (
        dataset_registry[
            dataset_name
        ]
    )

    embeddings = load_embeddings(
        dataset_info["embedding_path"],
        standardize=STANDARDIZE_EMBEDDINGS
    )

    if embeddings.shape[0] != len(node_order):
        raise ValueError(
            f"{dataset_name}: embedding rows "
            f"({embeddings.shape[0]}) differ from "
            f"the node order ({len(node_order)})."
        )

    return embeddings


loaded_embeddings = {}

embedding_dimensions = set()

for dataset_name in dataset_registry:
    embeddings = load_dataset_from_registry(
        dataset_name
    )

    loaded_embeddings[
        dataset_name
    ] = embeddings

    embedding_dimensions.add(
        embeddings.shape[1]
    )

    print(
        dataset_registry[dataset_name]["label"],
        embeddings.shape
    )

if len(embedding_dimensions) != 1:
    raise ValueError(
        "The embedding matrices do not have "
        "the same embedding dimension."
    )

print(
    "All directed embeddings are aligned with "
    "the Step 3 node order."
)

In [ ]:
# ------------------------------------------------------------
# 9. Internal and external clustering metrics
# ------------------------------------------------------------

def compute_internal_clustering_metrics(
    embeddings,
    labels
):
    n_samples = len(labels)

    n_clusters_found = len(
        np.unique(labels)
    )

    if (
        n_clusters_found < 2
        or n_clusters_found >= n_samples
    ):
        return {
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan,
            "n_clusters_found": n_clusters_found,
            "valid_internal_metrics": False,
        }

    return {
        "silhouette": silhouette_score(
            embeddings,
            labels
        ),
        "calinski_harabasz": (
            calinski_harabasz_score(
                embeddings,
                labels
            )
        ),
        "davies_bouldin": (
            davies_bouldin_score(
                embeddings,
                labels
            )
        ),
        "n_clusters_found": n_clusters_found,
        "valid_internal_metrics": True,
    }


def compute_hungarian_cluster_accuracy(
    true_labels,
    predicted_labels
):
    true_codes, true_classes = pd.factorize(
        true_labels,
        sort=True
    )

    predicted_codes, predicted_classes = pd.factorize(
        predicted_labels,
        sort=True
    )

    confusion = np.zeros(
        (
            len(predicted_classes),
            len(true_classes)
        ),
        dtype=int
    )

    for predicted_code, true_code in zip(
        predicted_codes,
        true_codes
    ):
        confusion[
            predicted_code,
            true_code
        ] += 1

    row_indices, column_indices = (
        linear_sum_assignment(
            -confusion
        )
    )

    matched_nodes = confusion[
        row_indices,
        column_indices
    ].sum()

    accuracy = (
        matched_nodes
        / len(true_labels)
    )

    return accuracy


def compute_external_clustering_metrics(
    true_labels,
    predicted_labels
):
    return {
        "adjusted_rand": adjusted_rand_score(
            true_labels,
            predicted_labels
        ),
        "normalized_mutual_information": (
            normalized_mutual_info_score(
                true_labels,
                predicted_labels
            )
        ),
        "adjusted_mutual_information": (
            adjusted_mutual_info_score(
                true_labels,
                predicted_labels
            )
        ),
        "homogeneity": homogeneity_score(
            true_labels,
            predicted_labels
        ),
        "completeness": completeness_score(
            true_labels,
            predicted_labels
        ),
        "v_measure": v_measure_score(
            true_labels,
            predicted_labels
        ),
        "hungarian_accuracy": (
            compute_hungarian_cluster_accuracy(
                true_labels,
                predicted_labels
            )
        ),
    }

In [ ]:
# ------------------------------------------------------------
# 10. Clustering models
# ------------------------------------------------------------

def build_clustering_model(
    method_name,
    n_clusters,
    random_state
):
    if method_name == "kmeans":
        return KMeans(
            n_clusters=n_clusters,
            random_state=random_state,
            n_init=10
        )

    if method_name == "gmm":
        return GaussianMixture(
            n_components=n_clusters,
            covariance_type="diag",
            reg_covar=1e-4,
            n_init=5,
            random_state=random_state
        )

    if method_name == "spectral":
        return SpectralClustering(
            n_clusters=n_clusters,
            affinity="rbf",
            assign_labels="kmeans",
            random_state=random_state
        )

    raise ValueError(
        f"Unknown clustering method: {method_name}"
    )


def fit_clustering_method_safely(
    method_name,
    embeddings,
    n_clusters,
    true_labels,
    random_state
):
    try:
        model = build_clustering_model(
            method_name=method_name,
            n_clusters=n_clusters,
            random_state=random_state
        )

        predicted_labels = model.fit_predict(
            embeddings
        )

        metrics = {
            "method": method_name,
            "k": n_clusters,
            "fit_status": "ok",
        }

        metrics.update(
            compute_internal_clustering_metrics(
                embeddings,
                predicted_labels
            )
        )

        metrics.update(
            compute_external_clustering_metrics(
                true_labels,
                predicted_labels
            )
        )

        if not metrics["valid_internal_metrics"]:
            metrics["fit_status"] = (
                "collapsed_or_invalid_number_of_clusters"
            )

        centroids = None

        if method_name == "kmeans":
            centroids = (
                model.cluster_centers_
            )

        return {
            "labels": predicted_labels,
            "metrics": metrics,
            "model": model,
            "centroids": centroids,
        }

    except Exception as error:
        return {
            "labels": None,
            "metrics": {
                "method": method_name,
                "k": n_clusters,
                "fit_status": (
                    f"error: {type(error).__name__}: {error}"
                ),
                "silhouette": np.nan,
                "calinski_harabasz": np.nan,
                "davies_bouldin": np.nan,
                "n_clusters_found": np.nan,
                "valid_internal_metrics": False,
                "adjusted_rand": np.nan,
                "normalized_mutual_information": np.nan,
                "adjusted_mutual_information": np.nan,
                "homogeneity": np.nan,
                "completeness": np.nan,
                "v_measure": np.nan,
                "hungarian_accuracy": np.nan,
            },
            "model": None,
            "centroids": None,
        }

In [ ]:
# ------------------------------------------------------------
# 11. Optional diagnostics for user-selected methods and k values
# ------------------------------------------------------------

def compute_clustering_diagnostics(
    embeddings,
    true_labels,
    methods,
    k_values,
    random_state=123
):
    diagnostic_rows = []

    for n_clusters in k_values:
        for method_name in methods:
            result = fit_clustering_method_safely(
                method_name=method_name,
                embeddings=embeddings,
                n_clusters=n_clusters,
                true_labels=true_labels,
                random_state=random_state
            )

            diagnostic_rows.append(
                result["metrics"]
            )

    return pd.DataFrame(
        diagnostic_rows
    )


def run_diagnostics_for_dataset(
    dataset_name,
    embeddings
):
    dataset_label = (
        dataset_registry[
            dataset_name
        ]["label"]
    )

    print(
        "\nRunning diagnostics for:",
        dataset_label
    )

    diagnostics_df = (
        compute_clustering_diagnostics(
            embeddings=embeddings,
            true_labels=ground_truth_labels,
            methods=DIAGNOSTIC_METHODS,
            k_values=DIAGNOSTIC_K_VALUES,
            random_state=RANDOM_STATE
        )
    )

    diagnostics_df.insert(
        0,
        "dataset",
        dataset_name
    )

    diagnostics_df.insert(
        1,
        "dataset_label",
        dataset_label
    )

    k_text = (
        f"{min(DIAGNOSTIC_K_VALUES)}_"
        f"{max(DIAGNOSTIC_K_VALUES)}"
        if len(DIAGNOSTIC_K_VALUES) > 0
        else "none"
    )

    maybe_save_csv(
        diagnostics_df,
        OUTPUT_FOLDER
        / f"{dataset_name}_diagnostics_k_{k_text}.csv",
        index=False
    )

    display(diagnostics_df)

    return diagnostics_df


In [ ]:
# ------------------------------------------------------------
# 12. Visualization helpers
# ------------------------------------------------------------

def get_dataset_label(
    dataset_name
):
    return dataset_registry[
        dataset_name
    ]["label"]


def compute_tsne(
    embeddings,
    random_state=123
):
    perplexity = min(
        30,
        max(
            5,
            (embeddings.shape[0] - 1) // 3
        )
    )

    tsne = TSNE(
        n_components=2,
        random_state=random_state,
        init="pca",
        learning_rate="auto",
        perplexity=perplexity
    )

    return tsne.fit_transform(
        embeddings
    )


def plot_tsne_clusters(
    embeddings_2d,
    labels,
    dataset_name,
    method_name,
    n_clusters
):
    plt.figure(figsize=(8, 6))

    plt.scatter(
        embeddings_2d[:, 0],
        embeddings_2d[:, 1],
        c=labels,
        cmap="tab10",
        s=12,
        alpha=0.8
    )

    plt.title(
        f"{get_dataset_label(dataset_name)} "
        f"— {method_name.upper()}, k={n_clusters}"
    )

    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.tight_layout()

    maybe_save_figure(
        OUTPUT_FOLDER
        / (
            f"{dataset_name}_{method_name}_"
            f"tsne_k{n_clusters}.png"
        )
    )

    plt.show()


def plot_cluster_sizes(
    labels,
    dataset_name,
    method_name,
    n_clusters
):
    cluster_counts = (
        pd.Series(labels)
        .value_counts()
        .sort_index()
    )

    plt.figure(figsize=(8, 5))

    sns.barplot(
        x=cluster_counts.index,
        y=cluster_counts.values
    )

    plt.xlabel("Cluster")
    plt.ylabel("Number of nodes")
    plt.title(
        f"{get_dataset_label(dataset_name)} "
        f"— {method_name.upper()}, k={n_clusters} "
        "— Cluster sizes"
    )

    plt.tight_layout()

    maybe_save_figure(
        OUTPUT_FOLDER
        / (
            f"{dataset_name}_{method_name}_"
            f"cluster_sizes_k{n_clusters}.png"
        )
    )

    plt.show()

    return cluster_counts


In [ ]:
# ------------------------------------------------------------
# 13. Class interpretation by cluster
# ------------------------------------------------------------

def compare_clusters_with_classes(
    class_membership_df,
    cluster_labels
):
    if (
        class_membership_df.shape[0]
        != len(cluster_labels)
    ):
        raise ValueError(
            "Class membership and cluster labels differ in length."
        )

    membership_with_cluster = (
        class_membership_df.copy()
    )

    membership_with_cluster[
        "cluster"
    ] = cluster_labels

    class_counts = (
        membership_with_cluster
        .groupby("cluster")
        .sum()
    )

    class_proportions = class_counts.div(
        class_counts.sum(axis=1),
        axis=0
    )

    return (
        class_counts,
        class_proportions
    )


def plot_class_composition(
    class_proportions,
    dataset_name,
    method_name,
    n_clusters
):
    class_proportions.plot(
        kind="bar",
        stacked=True,
        figsize=(12, 6),
        colormap="tab20"
    )

    plt.title(
        f"{get_dataset_label(dataset_name)} "
        f"— {method_name.upper()}, k={n_clusters} "
        "— Ground-truth class composition"
    )

    plt.xlabel("Cluster")
    plt.ylabel("Proportion of nodes")

    plt.legend(
        title="Cora class",
        bbox_to_anchor=(1.05, 1),
        loc="upper left"
    )

    plt.tight_layout()

    maybe_save_figure(
        OUTPUT_FOLDER
        / (
            f"{dataset_name}_{method_name}_"
            f"class_composition_k{n_clusters}.png"
        )
    )

    plt.show()


def plot_class_profile_heatmap(
    class_proportions,
    dataset_name,
    method_name,
    n_clusters
):
    plt.figure(figsize=(10, 6))

    sns.heatmap(
        class_proportions.T,
        cmap="Blues",
        annot=True,
        fmt=".2f"
    )

    plt.title(
        f"{get_dataset_label(dataset_name)} "
        f"— {method_name.upper()}, k={n_clusters} "
        "— Class prevalence by cluster"
    )

    plt.xlabel("Cluster")
    plt.ylabel("Cora class")
    plt.tight_layout()

    maybe_save_figure(
        OUTPUT_FOLDER
        / (
            f"{dataset_name}_{method_name}_"
            f"class_heatmap_k{n_clusters}.png"
        )
    )

    plt.show()


In [ ]:
# ------------------------------------------------------------
# 14. Mutual information and decision-tree interpretation
# ------------------------------------------------------------

def compute_mutual_information_classes(
    class_membership_df,
    cluster_labels
):
    mutual_information = (
        mutual_info_classif(
            class_membership_df.to_numpy(),
            cluster_labels,
            discrete_features=True,
            random_state=RANDOM_STATE
        )
    )

    return pd.Series(
        mutual_information,
        index=class_membership_df.columns,
        name="mutual_information"
    ).sort_values(
        ascending=False
    )


def train_decision_tree_cluster_explainer(
    class_membership_df,
    cluster_labels,
    max_depth=4,
    min_samples_leaf=10
):
    tree = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=RANDOM_STATE,
        class_weight="balanced"
    )

    tree.fit(
        class_membership_df,
        cluster_labels
    )

    predicted_clusters = tree.predict(
        class_membership_df
    )

    tree_metrics = {
        "tree_fidelity_balanced_accuracy": (
            balanced_accuracy_score(
                cluster_labels,
                predicted_clusters
            )
        ),
        "tree_depth": tree.get_depth(),
        "n_leaves": tree.get_n_leaves(),
    }

    tree_report = classification_report(
        cluster_labels,
        predicted_clusters,
        output_dict=True,
        zero_division=0
    )

    tree_rules = export_text(
        tree,
        feature_names=list(
            class_membership_df.columns
        )
    )

    return (
        tree,
        tree_metrics,
        tree_report,
        tree_rules
    )


def plot_decision_tree_cluster_explainer(
    tree,
    class_membership_df,
    dataset_name,
    method_name,
    n_clusters
):
    plt.figure(figsize=(24, 12))

    plot_tree(
        tree,
        feature_names=list(
            class_membership_df.columns
        ),
        class_names=[
            str(class_label)
            for class_label in tree.classes_
        ],
        filled=True,
        rounded=True,
        fontsize=8
    )

    plt.title(
        f"{get_dataset_label(dataset_name)} "
        f"— {method_name.upper()}, k={n_clusters} "
        "— Decision-tree explanation"
    )

    plt.tight_layout()

    maybe_save_figure(
        OUTPUT_FOLDER
        / (
            f"{dataset_name}_{method_name}_"
            f"decision_tree_k{n_clusters}.png"
        )
    )

    plt.show()


In [ ]:
# ------------------------------------------------------------
# 15. Save cluster labels
# ------------------------------------------------------------

def save_cluster_labels(
    cluster_labels,
    dataset_name,
    method_name,
    n_clusters
):
    labels_df = pd.DataFrame(
        {
            "node_id": node_order,
            "node_position": np.arange(
                1,
                len(node_order) + 1
            ),
            "cluster": cluster_labels,
            "ground_truth_class": ground_truth_labels,
        }
    )

    maybe_save_csv(
        labels_df,
        OUTPUT_FOLDER
        / (
            f"{dataset_name}_{method_name}_"
            f"labels_k{n_clusters}.csv"
        ),
        index=False
    )

    return labels_df


In [ ]:
# ------------------------------------------------------------
# 16. Full final analysis for one embedding configuration
# ------------------------------------------------------------

def run_final_dataset_analysis(
    dataset_name,
    embeddings,
    method_name,
    n_clusters
):
    method_name = str(
        method_name
    ).lower()

    n_clusters = int(
        n_clusters
    )

    print("\n" + "=" * 80)
    print(
        "FINAL ANALYSIS:",
        get_dataset_label(dataset_name)
    )
    print(
        f"Method: {method_name} | k={n_clusters}"
    )
    print("=" * 80)

    final_result = fit_clustering_method_safely(
        method_name=method_name,
        embeddings=embeddings,
        n_clusters=n_clusters,
        true_labels=ground_truth_labels,
        random_state=RANDOM_STATE
    )

    if final_result["labels"] is None:
        raise RuntimeError(
            final_result["metrics"]["fit_status"]
        )

    cluster_labels = (
        final_result["labels"]
    )

    final_metrics_df = pd.DataFrame(
        [final_result["metrics"]]
    )

    print(
        f"\n=== Final {method_name.upper()} metrics "
        f"(k={n_clusters}) ==="
    )

    display(final_metrics_df)

    maybe_save_csv(
        final_metrics_df,
        OUTPUT_FOLDER
        / (
            f"{dataset_name}_{method_name}_"
            f"final_metrics_k{n_clusters}.csv"
        ),
        index=False
    )

    labels_df = save_cluster_labels(
        cluster_labels=cluster_labels,
        dataset_name=dataset_name,
        method_name=method_name,
        n_clusters=n_clusters
    )

    cluster_counts = plot_cluster_sizes(
        labels=cluster_labels,
        dataset_name=dataset_name,
        method_name=method_name,
        n_clusters=n_clusters
    )

    maybe_save_csv(
        cluster_counts.rename("node_count"),
        OUTPUT_FOLDER
        / (
            f"{dataset_name}_{method_name}_"
            f"cluster_sizes_k{n_clusters}.csv"
        )
    )

    embeddings_2d = compute_tsne(
        embeddings,
        random_state=RANDOM_STATE
    )

    plot_tsne_clusters(
        embeddings_2d=embeddings_2d,
        labels=cluster_labels,
        dataset_name=dataset_name,
        method_name=method_name,
        n_clusters=n_clusters
    )

    (
        class_counts,
        class_proportions
    ) = compare_clusters_with_classes(
        class_membership_df,
        cluster_labels
    )

    print("\n=== Ground-truth class counts per cluster ===")
    display(class_counts)

    print("\n=== Ground-truth class proportions per cluster ===")
    display(class_proportions)

    maybe_save_csv(
        class_counts,
        OUTPUT_FOLDER
        / (
            f"{dataset_name}_{method_name}_"
            f"class_counts_k{n_clusters}.csv"
        )
    )

    maybe_save_csv(
        class_proportions,
        OUTPUT_FOLDER
        / (
            f"{dataset_name}_{method_name}_"
            f"class_proportions_k{n_clusters}.csv"
        )
    )

    plot_class_composition(
        class_proportions=class_proportions,
        dataset_name=dataset_name,
        method_name=method_name,
        n_clusters=n_clusters
    )

    plot_class_profile_heatmap(
        class_proportions=class_proportions,
        dataset_name=dataset_name,
        method_name=method_name,
        n_clusters=n_clusters
    )

    mutual_information = (
        compute_mutual_information_classes(
            class_membership_df,
            cluster_labels
        )
    )

    print("\n=== Mutual information between classes and clusters ===")
    display(mutual_information)

    maybe_save_csv(
        mutual_information,
        OUTPUT_FOLDER
        / (
            f"{dataset_name}_{method_name}_"
            f"class_mutual_information_k{n_clusters}.csv"
        )
    )

    (
        tree,
        tree_metrics,
        tree_report,
        tree_rules
    ) = train_decision_tree_cluster_explainer(
        class_membership_df,
        cluster_labels
    )

    print("\n=== Decision-tree fidelity ===")
    display(
        pd.DataFrame(
            [tree_metrics]
        )
    )

    print("\n=== Decision-tree rules ===")
    print(tree_rules)

    plot_decision_tree_cluster_explainer(
        tree=tree,
        class_membership_df=class_membership_df,
        dataset_name=dataset_name,
        method_name=method_name,
        n_clusters=n_clusters
    )

    # K-means and GMM have explicit cluster centers / means.
    cluster_representatives_df = None

    if method_name == "kmeans":
        cluster_representatives = (
            final_result["model"]
            .cluster_centers_
        )

        representative_name = "centroid"

    elif method_name == "gmm":
        cluster_representatives = (
            final_result["model"]
            .means_
        )

        representative_name = "component_mean"

    else:
        cluster_representatives = None
        representative_name = None

    if cluster_representatives is not None:
        cluster_representatives_df = pd.DataFrame(
            cluster_representatives,
            columns=[
                f"embedding_dim_{dimension}"
                for dimension in range(
                    cluster_representatives.shape[1]
                )
            ]
        )

        cluster_representatives_df.index.name = (
            "cluster"
        )

        maybe_save_csv(
            cluster_representatives_df,
            OUTPUT_FOLDER
            / (
                f"{dataset_name}_{method_name}_"
                f"{representative_name}s_k{n_clusters}.csv"
            )
        )

    return {
        "method": method_name,
        "k": n_clusters,
        "metrics_df": final_metrics_df,
        "labels": cluster_labels,
        "labels_df": labels_df,
        "cluster_counts": cluster_counts,
        "class_counts": class_counts,
        "class_proportions": class_proportions,
        "mutual_information": mutual_information,
        "tree": tree,
        "tree_metrics": tree_metrics,
        "tree_report": tree_report,
        "tree_rules": tree_rules,
        "cluster_representatives_df": cluster_representatives_df,
    }


In [ ]:
# ------------------------------------------------------------
# 17. OPTIONAL diagnostic sweep
# ------------------------------------------------------------

all_diagnostics = {}
diagnostics_summary = pd.DataFrame()

if RUN_DIAGNOSTICS:
    for dataset_name, embeddings in loaded_embeddings.items():
        all_diagnostics[
            dataset_name
        ] = run_diagnostics_for_dataset(
            dataset_name=dataset_name,
            embeddings=embeddings
        )

    diagnostics_summary = pd.concat(
        all_diagnostics.values(),
        ignore_index=True
    )

    maybe_save_csv(
        diagnostics_summary,
        OUTPUT_FOLDER
        / "all_directed_clustering_diagnostics.csv",
        index=False
    )

    print(
        "All optional clustering diagnostics completed."
    )

else:
    print(
        "Diagnostic sweep skipped. "
        "Set RUN_DIAGNOSTICS = True if you want to explore "
        "other methods and k values."
    )


In [ ]:
# ------------------------------------------------------------
# 18. Optional diagnostic comparison table
# ------------------------------------------------------------

if RUN_DIAGNOSTICS:
    diagnostic_comparison = (
        diagnostics_summary[
            [
                "dataset",
                "dataset_label",
                "method",
                "k",
                "silhouette",
                "calinski_harabasz",
                "davies_bouldin",
                "adjusted_rand",
                "normalized_mutual_information",
                "hungarian_accuracy",
                "fit_status",
            ]
        ]
        .sort_values(
            [
                "dataset",
                "method",
                "k",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    print(
        "=== Diagnostic comparison ==="
    )

    display(
        diagnostic_comparison
    )

    maybe_save_csv(
        diagnostic_comparison,
        OUTPUT_FOLDER
        / "directed_cora_diagnostic_comparison.csv",
        index=False
    )

else:
    diagnostic_comparison = pd.DataFrame()


In [ ]:
# ------------------------------------------------------------
# 19. Run the MANUALLY CONFIGURED final analysis
# ------------------------------------------------------------

final_results = {}

for dataset_name, embeddings in loaded_embeddings.items():
    config = FINAL_CLUSTERING_CONFIG[
        dataset_name
    ]

    final_results[
        dataset_name
    ] = run_final_dataset_analysis(
        dataset_name=dataset_name,
        embeddings=embeddings,
        method_name=config["method"],
        n_clusters=config["k"]
    )

print(
    "All manually configured final analyses completed."
)


In [ ]:
# ------------------------------------------------------------
# 20. Final summary of the manually selected configurations
# ------------------------------------------------------------

summary_rows = []

for dataset_name, result in final_results.items():
    row = (
        result["metrics_df"]
        .iloc[0]
        .to_dict()
    )

    row["dataset"] = dataset_name
    row["dataset_label"] = (
        get_dataset_label(
            dataset_name
        )
    )

    summary_rows.append(
        row
    )

final_metrics_summary = pd.DataFrame(
    summary_rows
)

column_order = [
    "dataset",
    "dataset_label",
    "method",
    "k",
    "silhouette",
    "calinski_harabasz",
    "davies_bouldin",
    "adjusted_rand",
    "normalized_mutual_information",
    "adjusted_mutual_information",
    "homogeneity",
    "completeness",
    "v_measure",
    "hungarian_accuracy",
    "n_clusters_found",
    "fit_status",
]

final_metrics_summary = (
    final_metrics_summary[
        column_order
    ]
)

print(
    "=== Final manually configured clustering summary ==="
)

display(
    final_metrics_summary
)

maybe_save_csv(
    final_metrics_summary,
    OUTPUT_FOLDER
    / "summary_manual_directed_clustering_configurations.csv",
    index=False
)


In [ ]:
# ------------------------------------------------------------
# 20A. Direct comparison:
#      post-training vs training-time embeddings
# ------------------------------------------------------------

comparison_pairs = [
    (
        "One Encoder",
        "one_encoder_directed",
        "one_encoder_training_directed"
    ),
    (
        "Full Batch",
        "batch_directed",
        "batch_training_directed"
    ),
]

comparison_rows = []

for (
    approach_label,
    post_dataset,
    training_dataset
) in comparison_pairs:

    post_metrics = (
        final_results[
            post_dataset
        ]["metrics_df"]
        .iloc[0]
        .to_dict()
    )

    training_metrics = (
        final_results[
            training_dataset
        ]["metrics_df"]
        .iloc[0]
        .to_dict()
    )

    for source_name, metrics in [
        ("post_training_reencoded", post_metrics),
        ("training_time", training_metrics),
    ]:
        comparison_rows.append({
            "approach": approach_label,
            "embedding_source": source_name,
            "method": metrics["method"],
            "k": metrics["k"],
            "silhouette": metrics["silhouette"],
            "calinski_harabasz": metrics["calinski_harabasz"],
            "davies_bouldin": metrics["davies_bouldin"],
            "adjusted_rand": metrics["adjusted_rand"],
            "normalized_mutual_information": (
                metrics["normalized_mutual_information"]
            ),
            "adjusted_mutual_information": (
                metrics["adjusted_mutual_information"]
            ),
            "homogeneity": metrics["homogeneity"],
            "completeness": metrics["completeness"],
            "v_measure": metrics["v_measure"],
            "hungarian_accuracy": metrics["hungarian_accuracy"],
            "n_clusters_found": metrics["n_clusters_found"],
            "fit_status": metrics["fit_status"],
        })

training_vs_post_comparison = pd.DataFrame(
    comparison_rows
)

print(
    "=== DIRECTED CORA: POST-TRAINING VS TRAINING-TIME ==="
)

display(
    training_vs_post_comparison
)

maybe_save_csv(
    training_vs_post_comparison,
    OUTPUT_FOLDER
    / "training_vs_post_training_clustering_comparison.csv",
    index=False
)


In [ ]:
# ------------------------------------------------------------
# 21. Final validation
# ------------------------------------------------------------

expected_number_of_nodes = len(
    node_order
)

for dataset_name, embeddings in loaded_embeddings.items():
    if (
        embeddings.shape[0]
        != expected_number_of_nodes
    ):
        raise ValueError(
            f"{dataset_name} has an incorrect number of rows."
        )

    if not np.isfinite(
        embeddings
    ).all():
        raise ValueError(
            f"{dataset_name} contains non-finite embeddings."
        )

if set(final_results) != set(
    dataset_registry
):
    raise ValueError(
        "Not all directed embedding configurations "
        "were analysed."
    )

for dataset_name, result in final_results.items():
    expected_method = (
        FINAL_CLUSTERING_CONFIG[
            dataset_name
        ]["method"]
    )

    expected_k = (
        FINAL_CLUSTERING_CONFIG[
            dataset_name
        ]["k"]
    )

    if result["method"] != expected_method:
        raise ValueError(
            f"{dataset_name}: unexpected clustering method."
        )

    if result["k"] != expected_k:
        raise ValueError(
            f"{dataset_name}: unexpected k."
        )

print("Step 4 completed successfully.")
print("Citation orientation validated: citing -> cited")
print("Outgoing ARD validated against corrected graph out-degrees.")
print("Nodes analysed:", expected_number_of_nodes)
print("Ground-truth classes:", class_membership_df.shape[1])
print("Embedding configurations:", len(dataset_registry))
print("Training-time comparison included for One Encoder and Full Batch.")
print("Results folder:", OUTPUT_FOLDER)

print("\n=== CONFIGURATIONS USED ===")

for dataset_name, result in final_results.items():
    print(
        f"{get_dataset_label(dataset_name):25s} "
        f"method={result['method']:8s} "
        f"k={result['k']}"
    )
